# 8.3 Edge Deployment Lab: GGUF Quantization Tradeoffs[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.3_edge_deployment/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.3_edge_deployment/lab.ipynb)Measure how quantization level affects model size, memory footprint, and throughput on CPU.We simulate GGUF characteristics without requiring llama.cpp installation.

In [ ]:
# -- Cell 1: Install dependencies via subprocess (no shell commands) --import subprocess, sys# Install numpy and matplotlib for plottingsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])import numpy as np  # Numerical computation libraryimport matplotlib.pyplot as plt  # Plotting library# Set random seed for reproducibilitynp.random.seed(42)# Use clean plot styleplt.style.use('seaborn-v0_8-whitegrid')print("Setup complete.")

## Experiment 1: Quantization Compression RatiosGGUF quantization compresses FP16 weights to fewer bits. Each level trades quality for size.We compute theoretical and measured compression for a 7B parameter model.

In [ ]:
# -- Cell 2: Compute GGUF size estimates for 7B model --# Total parameters in a 7B modeltotal_params = 7_000_000_000  # 7 billion weights# Bits per weight for each quantization level (from GGUF spec)quant_bits = {    "F16": 16.0,    # Full 16-bit floating point (baseline)    "Q8_0": 8.0,    # 8-bit uniform quantization    "Q6_K": 6.0,    # 6-bit k-quant method    "Q5_K_M": 5.5,  # 5-bit k-quant medium quality    "Q4_K_M": 4.5,  # 4-bit k-quant medium (most popular)    "Q3_K_M": 3.5,  # 3-bit k-quant medium    "Q2_K": 2.6,    # 2-bit k-quant (extreme compression)}# Calculate file size in GB for each quantization level# Formula: (params * bits_per_weight) / 8 bytes / 1e9 GB# Dictionary to store computed sizes for each quant levelsizes_gb = {}# Iterate over each quantization level to compute file sizefor quant, bits in quant_bits.items():    # Convert bits to bytes, then to gigabytes    size_bytes = total_params * bits / 8    sizes_gb[quant] = size_bytes / 1e9# Display results as a formatted tableprint(f"{'Quant':<10} {'Bits':<6} {'Size (GB)':<10} {'Compression vs F16':<20}")print("-" * 46)# Reference size is FP16 (baseline)fp16_size = sizes_gb["F16"]for quant, size in sizes_gb.items():    # Compression ratio = baseline / compressed size    ratio = fp16_size / size    print(f"{quant:<10} {quant_bits[quant]:<6.1f} {size:<10.2f} {ratio:<20.1f}x")

In [ ]:
# -- Cell 3: Visualize size vs quality tradeoff --# Quality scores (approximate perplexity retention, 1.0 = perfect)# Based on published GGUF benchmarks for Llama 7Bquality_retention = {    "F16": 1.000,    # Perfect baseline    "Q8_0": 0.998,   # Nearly lossless    "Q6_K": 0.995,   # Excellent    "Q5_K_M": 0.990, # Very good    "Q4_K_M": 0.980, # Good (recommended default)    "Q3_K_M": 0.960, # Noticeable degradation    "Q2_K": 0.920,   # Significant quality loss}# Create figure with two y-axes for size and qualityfig, ax1 = plt.subplots(figsize=(10, 5))ax2 = ax1.twinx()  # Second y-axis shares the x-axis# X positions for bar chartquants = list(quant_bits.keys())x = np.arange(len(quants))# Plot model size as bars (left y-axis)bars = ax1.bar(x, [sizes_gb[q] for q in quants], color="#dbeafe", edgecolor="#1e293b", alpha=0.8)ax1.set_ylabel("Model Size (GB)", color="#1e293b")ax1.set_ylim(0, 16)# Plot quality retention as line (right y-axis)quality_vals = [quality_retention[q] for q in quants]ax2.plot(x, quality_vals, "ro-", linewidth=2, markersize=8, label="Quality Retention")ax2.set_ylabel("Quality Retention (1.0 = FP16)", color="red")ax2.set_ylim(0.9, 1.01)# Highlight Q4_K_M as the sweet spotsweet_spot_idx = quants.index("Q4_K_M")bars[sweet_spot_idx].set_color("#dcfce7")  # Green highlightbars[sweet_spot_idx].set_edgecolor("#166534")# Labels and formattingax1.set_xticks(x)ax1.set_xticklabels(quants, rotation=45)ax1.set_title("GGUF Quantization: Size vs Quality (7B Model)")ax2.legend(loc="upper right")plt.tight_layout()plt.show()# Key insight: Q4_K_M reduces size by 3.6x with only 2% quality lossprint("Sweet spot: Q4_K_M gives 3.6x compression with 98% quality retention")

## Experiment 2: Simulated Throughput by PlatformDifferent hardware achieves different token rates due to memory bandwidth and compute limits.We model expected throughput based on the bandwidth-bound decode equation.

In [ ]:
# -- Cell 4: Model throughput from memory bandwidth --# Memory bandwidth for each platform (GB/s)platforms = {    "Raspberry Pi 5": {"bandwidth_gbs": 8.5, "ram_gb": 8},    "Laptop CPU (DDR5)": {"bandwidth_gbs": 50, "ram_gb": 32},    "M1 Max": {"bandwidth_gbs": 400, "ram_gb": 64},    "M2 Ultra": {"bandwidth_gbs": 800, "ram_gb": 192},    "Jetson Orin (GPU)": {"bandwidth_gbs": 204, "ram_gb": 64},}# For decode: each token requires loading all model weights once# Throughput (tok/s) = bandwidth / model_size_bytes# Using Q4_K_M (4.1 GB for 7B model)model_size_bytes = sizes_gb["Q4_K_M"] * 1e9  # Convert GB to bytesprint(f"{'Platform':<22} {'Bandwidth':<12} {'Est. tok/s':<12} {'Fits 7B Q4?':<12}")print("-" * 58)throughputs = {}for platform, specs in platforms.items():    # Theoretical max: bandwidth / weight_bytes_per_token_step    # Apply 60% efficiency factor for real-world overhead    efficiency = 0.60    theoretical_tps = (specs["bandwidth_gbs"] * 1e9 / model_size_bytes) * efficiency    # Check if model fits in RAM (need model + KV cache + runtime)    fits = specs["ram_gb"] >= 8  # Q4_K_M 7B needs ~8 GB total    throughputs[platform] = theoretical_tps if fits else 0    fit_str = "Yes" if fits else "No (too small)"    print(f"{platform:<22} {specs['bandwidth_gbs']:<12} {theoretical_tps:<12.1f} {fit_str:<12}")

In [ ]:
# -- Cell 5: Visualize platform throughput comparison --fig, ax = plt.subplots(figsize=(10, 5))# Sort platforms by throughput for clean visualizationsorted_platforms = sorted(throughputs.items(), key=lambda x: x[1])names = [p[0] for p in sorted_platforms]tps_values = [p[1] for p in sorted_platforms]# Horizontal bar chart (easier to read platform names)colors = ["#ffe4e6" if t < 10 else "#fef3c7" if t < 50 else "#dcfce7" for t in tps_values]ax.barh(names, tps_values, color=colors, edgecolor="#1e293b")# Add throughput labels on barsfor i, (name, tps) in enumerate(sorted_platforms):    ax.text(tps + 2, i, f"{tps:.0f} tok/s", va="center", fontsize=10)# Mark the "interactive threshold" (6-7 tok/s = reading speed)ax.axvline(7, color="red", linestyle="--", linewidth=1.5, label="Reading speed (7 tok/s)")ax.set_xlabel("Estimated Tokens/Second (7B Q4_K_M)")ax.set_title("Edge Platform Throughput (Bandwidth-Limited Decode)")ax.legend()plt.tight_layout()plt.show()# All platforms above red line can serve interactive chatprint("Any platform above the red line provides usable interactive chat speed.")

## Experiment 3: RAM Budget BreakdownOn edge devices, RAM is the binding constraint. Understanding the breakdown betweenmodel weights, KV cache, and runtime overhead determines which models fit.

In [ ]:
# -- Cell 6: RAM budget analysis for different device classes --def compute_ram_budget(model_size_gb, context_length, kv_bytes_per_token):    """Calculate total RAM needed for edge inference."""    # Model weights (already quantized size)    weight_memory = model_size_gb    # KV cache: stores key/value pairs for all tokens in context    kv_cache_gb = (context_length * kv_bytes_per_token) / 1e9    # Runtime overhead: tokenizer, scratch buffers, OS    runtime_gb = 0.5  # Approximately 500 MB fixed overhead    # Total RAM requirement    total = weight_memory + kv_cache_gb + runtime_gb    return {"weights": weight_memory, "kv_cache": kv_cache_gb, "runtime": runtime_gb, "total": total}# KV cache per token for 7B model at Q4: ~256 KB (32 layers, 32 heads, 128 dim, K+V)kv_per_token = 32 * 32 * 128 * 2 * 2  # layers * heads * dim * K+V * bytes(FP16)# Compute for different context sizes# Test multiple context lengths to show how KV cache scalescontext_sizes = [512, 1024, 2048, 4096, 8192]# Use Q4_K_M as the representative model sizemodel_q4_size = sizes_gb["Q4_K_M"]fig, ax = plt.subplots(figsize=(10, 5))# Stacked bar chart showing memory breakdownweights_vals = []kv_vals = []runtime_vals = []for ctx in context_sizes:    budget = compute_ram_budget(model_q4_size, ctx, kv_per_token)    weights_vals.append(budget["weights"])    kv_vals.append(budget["kv_cache"])    runtime_vals.append(budget["runtime"])x = np.arange(len(context_sizes))# Stack the three componentsax.bar(x, weights_vals, label="Model Weights", color="#dbeafe", edgecolor="#1e293b")ax.bar(x, kv_vals, bottom=weights_vals, label="KV Cache", color="#fef3c7", edgecolor="#1e293b")bottoms = [w + k for w, k in zip(weights_vals, kv_vals)]ax.bar(x, runtime_vals, bottom=bottoms, label="Runtime", color="#f3f4f6", edgecolor="#1e293b")# Draw RAM capacity lines for common devicesax.axhline(8, color="red", linestyle="--", alpha=0.7, label="8 GB device")ax.axhline(16, color="orange", linestyle="--", alpha=0.7, label="16 GB device")ax.set_xticks(x)ax.set_xticklabels([f"{c}" for c in context_sizes])ax.set_xlabel("Context Length (tokens)")ax.set_ylabel("Total RAM (GB)")ax.set_title("RAM Budget: 7B Q4_K_M Model by Context Length")ax.legend(loc="upper left")plt.tight_layout()plt.show()# Key finding: 8 GB devices can run 7B Q4 with up to ~2048 contextprint("8 GB RAM supports 7B Q4_K_M with up to ~2048 token context.")print("For 8192 context, you need 16+ GB RAM.")

## Key Takeaways1. **Q4_K_M is the sweet spot**: 3.6x compression with 98% quality retention2. **Bandwidth determines throughput**: Apple UMA (400-800 GB/s) dominates edge inference speed3. **RAM is the binding constraint**: Model + KV cache + runtime must fit in available memory4. **Context length costs scale linearly**: Each additional token costs fixed KV cache bytes5. **Any platform above 7 tok/s is interactive**: Matches human reading speed for chat